
# ColBERT Retrieval Swap Evaluation for MS-MARCO Abstractive QA

This notebook mirrors the Open-Domain QA ColBERT notebook, but evaluates on **MS-MARCO abstractive QA** using **ROUGE-L** and **BLEU-1**.

Goal:

- Compare **ColBERT + BART** against **DPR + BART** on the same MS-MARCO dev subset.
- Keep the generator fixed and swap only the retriever.
- Save predictions and summaries to an output folder so partial results are not lost.

This is an **evaluation-first** notebook. It does not train ColBERT or retrain RAG. Use it to test whether ColBERT retrieval improves abstractive QA generation.


In [ ]:

# Cell 1 — setup Drive, packages, project path, and local cache

!pip install -q transformers accelerate faiss-cpu sentencepiece datasets huggingface_hub pylate pyarrow tqdm

import os
import sys
import json
import glob
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import faiss
from tqdm.auto import tqdm

from google.colab import drive

# Avoid TensorFlow import noise in transformers.
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_TF"] = "0"

# Keep HuggingFace cache local, not in Drive.
os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["HF_DATASETS_CACHE"] = "/content/hf_cache/datasets"
os.environ["TRANSFORMERS_CACHE"] = "/content/hf_cache/transformers"
os.environ["FAISS_NO_AVX2"] = "1"

!mkdir -p /content/hf_cache/datasets /content/hf_cache/transformers

# Start from /content so Drive remounts cleanly.
os.chdir("/content")
try:
    drive.flush_and_unmount()
except Exception as e:
    print("Unmount warning:", e)

drive.mount("/content/gdrive", force_remount=True)

# Project path candidates. The first existing path is used.
PROJECT_CANDIDATES = [
    "/content/gdrive/MyDrive/JuniorYear/CS4782/final-proj",
    "/content/gdrive/MyDrive/Colab Notebooks/final-proj",
    "/content/gdrive/.shortcut-targets-by-id/1o8yF586YhxDQB4V3dTdjn4u47_bCq-qW/final-proj",
]

PROJECT_DIR = None
for p in PROJECT_CANDIDATES:
    if os.path.exists(p):
        PROJECT_DIR = p
        break

if PROJECT_DIR is None:
    raise FileNotFoundError("Could not find final-proj. Edit PROJECT_CANDIDATES in Cell 1.")

os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

print("Now working in:", os.getcwd())
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU is not enabled. Go to Runtime → Change runtime type → GPU before full evaluation.")


In [ ]:

# Cell 2 — verify required project files and MS-MARCO data

required_files = [
    "data/msmarco_dev.jsonl",
]

print("Required files:")
for f in required_files:
    print(f, "✅" if os.path.exists(f) else "❌")

print("\nCurrent data folder:")
!ls -lh data 2>/dev/null || true

# If MS-MARCO is missing, run Cell 3 to prepare it.


In [ ]:

# Cell 3 — optional: prepare cleaned MS-MARCO data if missing
#
# Skip this cell if data/msmarco_train.jsonl and data/msmarco_dev.jsonl already exist.
# This removes "No Answer Present" examples, which caused the earlier bad outputs.

%%writefile prepare_msmarco_colbert_eval.py
import argparse
import json
import os
from datasets import load_dataset


def is_no_answer(text):
    t = str(text).strip().lower()
    t = t.replace(".", "").replace("[", "").replace("]", "").strip()
    return t in {"", "no answer present", "no answer", "none"}


def clean_answers(x):
    if x is None:
        return []

    if isinstance(x, str):
        x = x.strip()
        return [] if is_no_answer(x) else [x]

    if isinstance(x, (list, tuple)):
        out = []
        for a in x:
            a = str(a).strip()
            if not is_no_answer(a):
                out.append(a)
        return out

    return []


def convert_split(ds, out_path, max_examples=None):
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    kept = 0

    with open(out_path, "w", encoding="utf-8") as f:
        for ex in ds:
            question = ex.get("query") or ex.get("question")
            if not question:
                continue

            # Prefer full-sentence MS-MARCO NLG answers.
            answers = clean_answers(ex.get("wellFormedAnswers"))
            if not answers:
                answers = clean_answers(ex.get("answers"))

            if not answers:
                continue

            row = {"question": question, "answers": answers}
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
            kept += 1

            if max_examples is not None and kept >= max_examples:
                break

    print(f"Wrote {kept:,} examples to {out_path}")


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--max_train", type=int, default=20000)
    parser.add_argument("--max_dev", type=int, default=1000)
    args = parser.parse_args()

    print("Loading MS-MARCO v2.1 from HuggingFace...")
    ds = load_dataset("ms_marco", "v2.1")

    convert_split(ds["train"], "data/msmarco_train.jsonl", max_examples=args.max_train)
    convert_split(ds["validation"], "data/msmarco_dev.jsonl", max_examples=args.max_dev)


if __name__ == "__main__":
    main()


In [ ]:

# Cell 4 — run MS-MARCO preparation only if needed

if os.path.exists("data/msmarco_dev.jsonl"):
    print("MS-MARCO dev file already exists. Skipping prep.")
    !ls -lh data/msmarco_train.jsonl data/msmarco_dev.jsonl 2>/dev/null || true
else:
    !python prepare_msmarco_colbert_eval.py --max_train 20000 --max_dev 1000

# Sanity check: these should print nothing.
!grep -i "No Answer Present" data/msmarco_train.jsonl | head || true
!grep -i "No Answer Present" data/msmarco_dev.jsonl | head || true


In [ ]:

# Cell 5 — load or create the same 50k passage sample used for ColBERT/DPR comparison
#
# This keeps ColBERT and DPR fair: both search over the exact same 50k passage set.

from huggingface_hub import hf_hub_download

WORK_DIR = Path("colbert_msmarco_eval")
WORK_DIR.mkdir(exist_ok=True)

META_PATH = WORK_DIR / "passages_50k_meta.parquet"
EMB_PATH = WORK_DIR / "passages_50k_dpr_embeddings.npy"

SUBSET_SIZE = 50_000
NUM_WIKIDPR_SHARDS = 4
RANDOM_SEED = 42

if META_PATH.exists() and EMB_PATH.exists():
    print("Loading cached 50k passage sample...")
    passages_sample = pd.read_parquet(META_PATH)
    dpr_embeddings = np.load(EMB_PATH).astype(np.float32)
else:
    print("Downloading/loading Wiki-DPR shards and creating a 50k sample...")
    shard_files = []
    for i in range(NUM_WIKIDPR_SHARDS):
        f = hf_hub_download(
            repo_id="facebook/wiki_dpr",
            filename=f"data/psgs_w100/nq/train-{i:05d}-of-00157.parquet",
            repo_type="dataset",
            cache_dir="/content/hf_cache",
        )
        shard_files.append(f)

    dfs = [pd.read_parquet(f) for f in shard_files]
    passages_df = pd.concat(dfs, ignore_index=True)
    print("Available columns:", list(passages_df.columns))

    passages_sample_full = passages_df.sample(n=SUBSET_SIZE, random_state=RANDOM_SEED).reset_index(drop=True)
    dpr_embeddings = np.vstack(passages_sample_full["embeddings"].to_numpy()).astype(np.float32)

    passages_sample = passages_sample_full[["id", "title", "text"]].copy().reset_index(drop=True)
    passages_sample.to_parquet(META_PATH, index=False)
    np.save(EMB_PATH, dpr_embeddings)

print("passages_sample:", passages_sample.shape)
print("dpr_embeddings:", dpr_embeddings.shape)
print(passages_sample.head(2))


In [ ]:

# Cell 6 — build/load DPR FAISS baseline index on the same 50k passage sample

DPR_FAISS_PATH = WORK_DIR / "dpr_50k.faiss"

if DPR_FAISS_PATH.exists():
    print("Loading cached DPR FAISS index:", DPR_FAISS_PATH)
    dpr_index = faiss.read_index(str(DPR_FAISS_PATH))
else:
    print("Building DPR FAISS index...")
    dpr_index = faiss.IndexFlatIP(dpr_embeddings.shape[1])
    dpr_index.add(dpr_embeddings.astype(np.float32))
    faiss.write_index(dpr_index, str(DPR_FAISS_PATH))

print("DPR index vectors:", dpr_index.ntotal)


In [ ]:

# Cell 7 — build ColBERT PLAID index on the same 50k passage sample

from pylate import indexes, models, retrieve

if not torch.cuda.is_available():
    raise RuntimeError("GPU is strongly recommended for ColBERT indexing/evaluation. Enable GPU and rerun.")

COLBERT_INDEX_FOLDER = "/content/pylate_msmarco_index"
COLBERT_INDEX_NAME = "wiki_colbert_50k_msmarco"

# ColBERT documents use row-index ids so retrieval maps directly back to passages_sample rows.
documents = (passages_sample["title"].fillna("") + ". " + passages_sample["text"].fillna("")).tolist()
doc_ids = [str(i) for i in range(len(passages_sample))]

print("Loading ColBERT model...")
colbert_model = models.ColBERT(
    model_name_or_path="lightonai/colbertv2.0",
    device="cuda",
)

print(f"Encoding {len(documents):,} documents for ColBERT...")
doc_embeddings = colbert_model.encode(
    documents,
    batch_size=64,
    is_query=False,
    show_progress_bar=True,
)

print("Building PLAID index...")
colbert_index = indexes.PLAID(
    index_folder=COLBERT_INDEX_FOLDER,
    index_name=COLBERT_INDEX_NAME,
    override=True,
)
colbert_index.add_documents(
    documents_ids=doc_ids,
    documents_embeddings=doc_embeddings,
)

colbert_retriever = retrieve.ColBERT(index=colbert_index)
print("ColBERT index ready.")


In [ ]:

# Cell 8 — load DPR question encoder and the answer generator
#
# Default: use your fine-tuned MS-MARCO RAG generator if it exists.
# Fallback: use vblagoje/bart_lfqa.

from transformers import (
    DPRQuestionEncoder,
    DPRQuestionEncoderTokenizer,
    BartForConditionalGeneration,
    AutoTokenizer,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading DPR question encoder...")
q_tokenizer = DPRQuestionEncoderTokenizer.from_pretrained("facebook/dpr-question_encoder-single-nq-base")
q_encoder = DPRQuestionEncoder.from_pretrained("facebook/dpr-question_encoder-single-nq-base").to(DEVICE).eval()

# Try to use your fine-tuned MS-MARCO RAG generator from the 3-epoch run.
USE_FINETUNED_MSMARCO_GENERATOR = True
GENERATOR_CANDIDATES = [
    Path("outputs/rag_baseline_msmarco_50k_answerable_3ep/checkpoint-3000"),
    Path("outputs/rag_baseline_msmarco_50k_answerable_3ep/checkpoint-2000"),
    Path("outputs/rag_baseline_msmarco_50k_answerable_3ep/checkpoint-1000"),
    Path("outputs/rag_baseline_msmarco_50k_answerable/checkpoint-1000"),
]

GENERATOR_SOURCE = None
for p in GENERATOR_CANDIDATES:
    if p.exists():
        GENERATOR_SOURCE = p
        break

if USE_FINETUNED_MSMARCO_GENERATOR and GENERATOR_SOURCE is not None:
    print("Loading fine-tuned MS-MARCO RAG generator from:", GENERATOR_SOURCE)
    from train_msmarco import load_model_and_tokenizer

    rag_model, rag_tokenizer = load_model_and_tokenizer(str(GENERATOR_SOURCE))
    bart_model = rag_model.rag.generator.to(DEVICE).eval()
    bart_tokenizer = rag_tokenizer.generator
    GENERATOR_NAME = str(GENERATOR_SOURCE)
else:
    print("Fine-tuned MS-MARCO checkpoint not found. Falling back to vblagoje/bart_lfqa.")
    bart_model = BartForConditionalGeneration.from_pretrained("vblagoje/bart_lfqa").to(DEVICE).eval()
    bart_tokenizer = AutoTokenizer.from_pretrained("vblagoje/bart_lfqa")
    GENERATOR_NAME = "vblagoje/bart_lfqa"

print("Generator:", GENERATOR_NAME)
print("Models loaded on", DEVICE)


In [ ]:

# Cell 9 — define retrieval, generation, BLEU-1, ROUGE-L, and resumable eval helpers

import re
import math
from collections import Counter


def normalize_tokens(text):
    return re.findall(r"\w+", str(text).lower())


def lcs_len(a, b):
    """Longest common subsequence length for ROUGE-L."""
    dp = [0] * (len(b) + 1)
    for x in a:
        prev = 0
        for j, y in enumerate(b, start=1):
            temp = dp[j]
            if x == y:
                dp[j] = prev + 1
            else:
                dp[j] = max(dp[j], dp[j - 1])
            prev = temp
    return dp[-1]


def rouge_l_f1(pred, ref):
    pred_toks = normalize_tokens(pred)
    ref_toks = normalize_tokens(ref)
    if not pred_toks or not ref_toks:
        return 0.0
    lcs = lcs_len(pred_toks, ref_toks)
    precision = lcs / len(pred_toks)
    recall = lcs / len(ref_toks)
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)


def corpus_bleu1(preds, refs_list):
    """Corpus BLEU-1 with clipped unigram precision and brevity penalty."""
    clipped_total = 0
    pred_total = 0
    pred_len_total = 0
    ref_len_total = 0

    for pred, refs in zip(preds, refs_list):
        pred_toks = normalize_tokens(pred)
        ref_tokens_list = [normalize_tokens(r) for r in refs if str(r).strip()]
        if not pred_toks or not ref_tokens_list:
            continue

        pred_counts = Counter(pred_toks)
        best_ref_counts = Counter()
        best_overlap = -1
        best_ref_len = len(ref_tokens_list[0])

        for ref_toks in ref_tokens_list:
            ref_counts = Counter(ref_toks)
            overlap = sum((pred_counts & ref_counts).values())
            if overlap > best_overlap:
                best_overlap = overlap
                best_ref_counts = ref_counts
                best_ref_len = len(ref_toks)

        clipped_total += sum((pred_counts & best_ref_counts).values())
        pred_total += len(pred_toks)
        pred_len_total += len(pred_toks)
        ref_len_total += best_ref_len

    if pred_total == 0:
        return 0.0

    precision = clipped_total / pred_total
    if pred_len_total == 0:
        bp = 0.0
    elif pred_len_total > ref_len_total:
        bp = 1.0
    else:
        bp = math.exp(1 - ref_len_total / pred_len_total)

    return 100 * bp * precision


def get_question(ex):
    return ex.get("question") or ex.get("query") or ex.get("input")


def get_answers(ex):
    answers = ex.get("answers") or ex.get("all_answers") or ex.get("answer") or []
    if isinstance(answers, str):
        answers = [answers]
    return answers


def load_jsonl(path, max_examples=None):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
            if max_examples is not None and len(rows) >= max_examples:
                break
    return rows


@torch.no_grad()
def dpr_retrieve(question, k=5):
    inputs = q_tokenizer(
        question,
        return_tensors="pt",
        truncation=True,
        max_length=128,
    ).to(DEVICE)
    q_emb = q_encoder(**inputs).pooler_output.detach().cpu().numpy().astype(np.float32)
    scores, ids = dpr_index.search(q_emb, k)

    docs = []
    for score, idx in zip(scores[0], ids[0]):
        row = int(idx)
        p = passages_sample.iloc[row]
        docs.append({
            "score": float(score),
            "row": row,
            "title": str(p["title"]),
            "text": str(p["text"]),
        })
    return docs


@torch.no_grad()
def colbert_retrieve(question, k=5):
    q_emb = colbert_model.encode(
        [question],
        batch_size=1,
        is_query=True,
        show_progress_bar=False,
    )
    results = colbert_retriever.retrieve(queries_embeddings=q_emb, k=k)

    docs = []
    for r in results[0]:
        row = int(str(r["id"]))
        p = passages_sample.iloc[row]
        docs.append({
            "score": float(r.get("score", 0.0)),
            "row": row,
            "title": str(p["title"]),
            "text": str(p["text"]),
        })
    return docs


@torch.no_grad()
def generate_answer(question, docs, max_new_tokens=64, num_beams=4):
    # Keep context compact so BART input does not overflow.
    context_parts = []
    for d in docs:
        context_parts.append(f"Title: {d['title']}\nText: {d['text'][:700]}")
    context = "\n\n".join(context_parts)

    prompt = f"question: {question}\ncontext: {context}"
    enc = bart_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024,
    ).to(DEVICE)

    out = bart_model.generate(
        **enc,
        max_new_tokens=max_new_tokens,
        num_beams=num_beams,
        no_repeat_ngram_size=3,
        early_stopping=True,
    )
    return bart_tokenizer.decode(out[0], skip_special_tokens=True).strip()


def summarize_predictions(predictions):
    preds = [p["prediction"] for p in predictions]
    refs_list = [p["answers"] for p in predictions]

    rouge_scores = []
    for pred, refs in zip(preds, refs_list):
        best = max(rouge_l_f1(pred, ref) for ref in refs) if refs else 0.0
        rouge_scores.append(best)

    rouge_l = 100 * sum(rouge_scores) / max(len(rouge_scores), 1)
    bleu1 = corpus_bleu1(preds, refs_list)

    return {
        "total": len(predictions),
        "rouge_l": rouge_l,
        "bleu1": bleu1,
    }


def eval_pipeline_resumable(retrieve_fn, examples, k, label, pred_path, summary_path, max_new_tokens=64, num_beams=4):
    """Run eval and save one JSONL line per example so interrupted runs can resume."""
    pred_path = Path(pred_path)
    summary_path = Path(summary_path)
    pred_path.parent.mkdir(parents=True, exist_ok=True)

    if summary_path.exists():
        print(f"{label} summary already exists. Loading:", summary_path)
        with open(summary_path, "r", encoding="utf-8") as f:
            return json.load(f)

    existing = []
    if pred_path.exists():
        with open(pred_path, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    existing.append(json.loads(line))
        print(f"Resuming {label}: found {len(existing)} existing predictions.")

    start_idx = len(existing)
    predictions = existing[:]

    with open(pred_path, "a", encoding="utf-8") as out_f:
        for i in tqdm(range(start_idx, len(examples)), desc=label):
            ex = examples[i]
            q = get_question(ex)
            answers = get_answers(ex)
            docs = retrieve_fn(q, k=k)
            pred = generate_answer(q, docs, max_new_tokens=max_new_tokens, num_beams=num_beams)

            row = {
                "i": i,
                "question": q,
                "prediction": pred,
                "answers": answers,
                "top_docs": [
                    {"title": d["title"], "score": d["score"], "text": d["text"][:300]}
                    for d in docs[:3]
                ],
            }
            predictions.append(row)
            out_f.write(json.dumps(row, ensure_ascii=False) + "\n")
            out_f.flush()

    metrics = summarize_predictions(predictions)
    summary = {
        "label": label,
        "k": k,
        "num_examples": len(examples),
        "generator": GENERATOR_NAME,
        "predictions_file": str(pred_path),
        **metrics,
    }

    with open(summary_path, "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)

    print("=" * 80)
    print(label)
    print(f"Examples evaluated: {summary['total']}")
    print(f"ROUGE-L: {summary['rouge_l']:.2f}")
    print(f"BLEU-1:  {summary['bleu1']:.2f}")
    print("Saved predictions:", pred_path)
    print("Saved summary:", summary_path)

    return summary


In [ ]:

# Cell 10 — retrieval sanity check on a few MS-MARCO questions

msmarco_smoke = load_jsonl("data/msmarco_dev.jsonl", max_examples=5)

for ex in msmarco_smoke[:3]:
    q = get_question(ex)
    print("=" * 100)
    print("QUESTION:", q)
    print("REFERENCE:", get_answers(ex)[:1])

    print("\nTop ColBERT docs:")
    for d in colbert_retrieve(q, k=3):
        print(f"- score={d['score']:.3f} | {d['title']} | {d['text'][:160].replace(chr(10), ' ')}")

    print("\nTop DPR docs:")
    for d in dpr_retrieve(q, k=3):
        print(f"- score={d['score']:.3f} | {d['title']} | {d['text'][:160].replace(chr(10), ' ')}")


In [ ]:

# Cell 11 — 10-example smoke eval with BLEU-1 / ROUGE-L
#
# Run this before the full eval to make sure generation and metrics work.

RESULTS_DIR = Path("outputs/colbert_msmarco_abstractive_eval")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

K = 5
SMOKE_N = 10
smoke_examples = load_jsonl("data/msmarco_dev.jsonl", max_examples=SMOKE_N)

smoke_colbert = eval_pipeline_resumable(
    colbert_retrieve,
    smoke_examples,
    k=K,
    label=f"ColBERT+BART MS-MARCO smoke K={K}",
    pred_path=RESULTS_DIR / f"smoke_predictions_colbert_bart_k{K}_{SMOKE_N}.jsonl",
    summary_path=RESULTS_DIR / f"smoke_summary_colbert_bart_k{K}_{SMOKE_N}.json",
)

smoke_dpr = eval_pipeline_resumable(
    dpr_retrieve,
    smoke_examples,
    k=K,
    label=f"DPR+BART MS-MARCO smoke K={K}",
    pred_path=RESULTS_DIR / f"smoke_predictions_dpr_bart_k{K}_{SMOKE_N}.jsonl",
    summary_path=RESULTS_DIR / f"smoke_summary_dpr_bart_k{K}_{SMOKE_N}.json",
)

print("\nSmoke comparison:")
print(json.dumps({"colbert": smoke_colbert, "dpr": smoke_dpr}, indent=2))


In [ ]:

# Cell 12 — full configurable MS-MARCO abstractive eval
#
# Recommended settings:
# - Start with EVAL_MAX_EXAMPLES=500.
# - Use 1000 for the final reported number if time allows.
# - The output files are resumable, so interrupted runs can continue.

EVAL_MAX_EXAMPLES = 500   # change to 1000 for final reported eval
K = 5
NUM_BEAMS = 4
MAX_NEW_TOKENS = 64

RESULTS_DIR = Path("outputs/colbert_msmarco_abstractive_eval")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

msmarco_dev = load_jsonl("data/msmarco_dev.jsonl", max_examples=EVAL_MAX_EXAMPLES)
print(f"Loaded {len(msmarco_dev)} MS-MARCO dev examples")
print("Saving results to:", RESULTS_DIR)

run_config = {
    "task": "abstractive_qa",
    "dataset": "MS-MARCO v2.1 dev subset",
    "max_examples": EVAL_MAX_EXAMPLES,
    "k": K,
    "num_beams": NUM_BEAMS,
    "max_new_tokens": MAX_NEW_TOKENS,
    "generator": GENERATOR_NAME,
    "retrievers": ["ColBERT", "DPR/FAISS"],
    "metrics": ["ROUGE-L", "BLEU-1"],
}

with open(RESULTS_DIR / f"run_config_k{K}_{len(msmarco_dev)}.json", "w", encoding="utf-8") as f:
    json.dump(run_config, f, indent=2)

summary_colbert = eval_pipeline_resumable(
    colbert_retrieve,
    msmarco_dev,
    k=K,
    label=f"ColBERT+BART MS-MARCO K={K}",
    pred_path=RESULTS_DIR / f"predictions_colbert_bart_k{K}_{len(msmarco_dev)}.jsonl",
    summary_path=RESULTS_DIR / f"summary_colbert_bart_k{K}_{len(msmarco_dev)}.json",
    max_new_tokens=MAX_NEW_TOKENS,
    num_beams=NUM_BEAMS,
)

summary_dpr = eval_pipeline_resumable(
    dpr_retrieve,
    msmarco_dev,
    k=K,
    label=f"DPR+BART MS-MARCO K={K}",
    pred_path=RESULTS_DIR / f"predictions_dpr_bart_k{K}_{len(msmarco_dev)}.jsonl",
    summary_path=RESULTS_DIR / f"summary_dpr_bart_k{K}_{len(msmarco_dev)}.json",
    max_new_tokens=MAX_NEW_TOKENS,
    num_beams=NUM_BEAMS,
)

comparison = {
    "k": K,
    "num_examples": len(msmarco_dev),
    "generator": GENERATOR_NAME,
    "colbert": summary_colbert,
    "dpr": summary_dpr,
    "delta_rouge_l_colbert_minus_dpr": summary_colbert["rouge_l"] - summary_dpr["rouge_l"],
    "delta_bleu1_colbert_minus_dpr": summary_colbert["bleu1"] - summary_dpr["bleu1"],
}

comparison_path = RESULTS_DIR / f"comparison_k{K}_{len(msmarco_dev)}.json"
with open(comparison_path, "w", encoding="utf-8") as f:
    json.dump(comparison, f, indent=2)

print("=" * 80)
print("Final comparison:")
print(json.dumps(comparison, indent=2))
print("Saved comparison:", comparison_path)


In [ ]:

# Cell 13 — inspect qualitative predictions from both systems

RESULTS_DIR = Path("outputs/colbert_msmarco_abstractive_eval")

colbert_files = sorted(RESULTS_DIR.glob("predictions_colbert_bart_k*_*.jsonl"))
dpr_files = sorted(RESULTS_DIR.glob("predictions_dpr_bart_k*_*.jsonl"))

print("ColBERT prediction files:")
for f in colbert_files:
    print("-", f)

print("\nDPR prediction files:")
for f in dpr_files:
    print("-", f)

if colbert_files:
    print("\nColBERT examples:")
    !head -n 5 "{colbert_files[-1]}"

if dpr_files:
    print("\nDPR examples:")
    !head -n 5 "{dpr_files[-1]}"


In [ ]:

# Cell 14 — optional K sweep for MS-MARCO ROUGE-L / BLEU-1
#
# This can be slow because it runs generation repeatedly.
# Use small K_VALUES and EVAL_MAX_EXAMPLES_SWEEP first.

RUN_K_SWEEP = False

if RUN_K_SWEEP:
    K_VALUES = [1, 5, 10, 20]
    EVAL_MAX_EXAMPLES_SWEEP = 200
    NUM_BEAMS = 4
    MAX_NEW_TOKENS = 64

    sweep_examples = load_jsonl("data/msmarco_dev.jsonl", max_examples=EVAL_MAX_EXAMPLES_SWEEP)
    sweep_rows = []

    for K in K_VALUES:
        colbert_sum = eval_pipeline_resumable(
            colbert_retrieve,
            sweep_examples,
            k=K,
            label=f"ColBERT+BART sweep K={K}",
            pred_path=RESULTS_DIR / f"sweep_predictions_colbert_bart_k{K}_{len(sweep_examples)}.jsonl",
            summary_path=RESULTS_DIR / f"sweep_summary_colbert_bart_k{K}_{len(sweep_examples)}.json",
            max_new_tokens=MAX_NEW_TOKENS,
            num_beams=NUM_BEAMS,
        )
        dpr_sum = eval_pipeline_resumable(
            dpr_retrieve,
            sweep_examples,
            k=K,
            label=f"DPR+BART sweep K={K}",
            pred_path=RESULTS_DIR / f"sweep_predictions_dpr_bart_k{K}_{len(sweep_examples)}.jsonl",
            summary_path=RESULTS_DIR / f"sweep_summary_dpr_bart_k{K}_{len(sweep_examples)}.json",
            max_new_tokens=MAX_NEW_TOKENS,
            num_beams=NUM_BEAMS,
        )

        sweep_rows.append({
            "k": K,
            "colbert_rouge_l": colbert_sum["rouge_l"],
            "colbert_bleu1": colbert_sum["bleu1"],
            "dpr_rouge_l": dpr_sum["rouge_l"],
            "dpr_bleu1": dpr_sum["bleu1"],
        })

    sweep_df = pd.DataFrame(sweep_rows)
    sweep_path = RESULTS_DIR / f"k_sweep_msmarco_{len(sweep_examples)}.csv"
    sweep_df.to_csv(sweep_path, index=False)
    print(sweep_df)
    print("Saved:", sweep_path)

    import matplotlib.pyplot as plt

    plt.figure(figsize=(7, 4))
    plt.plot(sweep_df["k"], sweep_df["colbert_rouge_l"], marker="o", label="ColBERT ROUGE-L")
    plt.plot(sweep_df["k"], sweep_df["dpr_rouge_l"], marker="o", label="DPR ROUGE-L")
    plt.xlabel("K retrieved documents")
    plt.ylabel("ROUGE-L")
    plt.title("MS-MARCO ROUGE-L vs K")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

    plt.figure(figsize=(7, 4))
    plt.plot(sweep_df["k"], sweep_df["colbert_bleu1"], marker="o", label="ColBERT BLEU-1")
    plt.plot(sweep_df["k"], sweep_df["dpr_bleu1"], marker="o", label="DPR BLEU-1")
    plt.xlabel("K retrieved documents")
    plt.ylabel("BLEU-1")
    plt.title("MS-MARCO BLEU-1 vs K")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()
else:
    print("K sweep is disabled. Set RUN_K_SWEEP=True to run it.")



## How to interpret the results

Use Cell 12 for the main comparison.

- If **ColBERT+BART ROUGE-L/BLEU-1 > DPR+BART**, then ColBERT retrieval helped the abstractive QA pipeline.
- If ColBERT retrieves better-looking evidence but metrics do not improve, the generator may not be using the evidence effectively.
- If both systems are below your fine-tuned RAG-Sequence MS-MARCO result, report this as an **architecture/retriever swap experiment**, not as your main RAG-Sequence reproduction.

Suggested result wording:

> We evaluated a retrieval-swap architecture on MS-MARCO abstractive QA by replacing DPR/FAISS retrieval with ColBERT while keeping the generator fixed. We report ROUGE-L and BLEU-1 for ColBERT+BART and DPR+BART on the same dev subset.
